In [ ]:
!pip install -q confluent_kafka
!pip install -q fastavro
!pip install -q azure-eventhub
!pip install -q azure.identity
!pip install -q azure.schemaregistry

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic
from datetime import datetime, timedelta
import random
import io
import json
from pyspark.sql import Row
from azure.eventhub import EventData
from azure.eventhub.aio import EventHubProducerClient
from fastavro import parse_schema, schemaless_writer

In [ ]:
APP_EVENT_AVRO_SCHEMA = {
    "type": "record",
    "name": "AppEvent",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_ts", "type": "string"},
        {"name": "session_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "event_type", "type": "string"},
        {"name": "product_id", "type": "string"},
        {"name": "channel", "type": "string"},
    ],
}

PARSED_AVRO_SCHEMA = parse_schema(APP_EVENT_AVRO_SCHEMA)

In [ ]:
class FamiaStreamGenerator:
    def __init__(self, namespace, connection_string):
        self._namespace = namespace
        self._connection_string = connection_string
        self._admin_client = AdminClient(self.get_kafka_config)


    @property
    def get_kafka_config(self):
        return {
            "bootstrap.servers": f"{self._namespace}.servicebus.windows.net:9093",
            "sasl.mechanism": "PLAIN",
            "security.protocol": "SASL_SSL",
            "sasl.username": "$ConnectionString",
            "sasl.password": self._connection_string,
            "request.timeout.ms": "60000"
        }


    def create_topic(self, topic_name, num_partitions=1, replication_factor=1):
        new_topic = NewTopic(topic=topic_name, num_partitions=num_partitions, replication_factor=replication_factor)
        fs = self._admin_client.create_topics([new_topic])

        for topic, f in fs.items():
            try:
                f.result()
                print(f"Topic \"{topic}\" creado exitosamente")
            except Exception as e:
                print(f"Error al crear el topic \"{topic}\": {e}")


    def list_topics(self):
        topic_list = self._admin_client.list_topics().topics
        return topic_list


    def __generate_data(self, rows=50):

        app_events = []

        for i in range(1, rows + 1):
            ts = datetime(2026, 4, 3, 9, 0, 0) + timedelta(seconds=i*15)
            app_events.append(Row(
                event_id=f"APP-{i:06d}",
                event_ts=ts,
                session_id=f"S{(i % 160) + 1:05d}",
                customer_id=f"C{(i % 100) + 1:05d}",
                event_type=random.choice(["session_start", "view_product", "add_to_cart", "checkout", "purchase"]),
                product_id=f"P{(i % 120) + 1:04d}",
                channel=random.choice(["android", "ios", "webview"]),
            ))

        return app_events


    def __row_to_dict(self, row):

        event = row.asDict(recursive=True)

        if isinstance(event["event_ts"], datetime):
            event["event_ts"] = event["event_ts"].isoformat()

        return event


    def __serialize_json(self, event):

        return json.dumps(
            event,
            ensure_ascii=False
        ).encode("utf-8")


    def __serialize_avro(self, event):

        buffer = io.BytesIO()

        schemaless_writer(
            buffer,
            PARSED_AVRO_SCHEMA,
            event,
        )

        return buffer.getvalue()


    async def produce_events(self, format, topic, rows=50):
        events = self.__generate_data(rows)

        async with EventHubProducerClient.from_connection_string(
            conn_str=self._connection_string,
            eventhub_name=topic,
        ) as producer:

            for row in events:

                event = self.__row_to_dict(row)

                if format == "json":
                    payload = self.__serialize_json(event)
                    message = EventData(payload)
                    message.content_type = "application/json"
                elif format == "avro":
                    payload = self.__serialize_avro(event)
                    message = EventData(payload)
                    message.content_type = "avro/binary"
                else:
                    raise ValueError(
                        f"Formato no soportado: {format}"
                    )

                await producer.send_batch(
                    [message],
                    partition_key=event["session_id"],
                )